# RUN on BASE kernel!

## Meshing based on already segmented mask

- postprocessing of already generated mesh
- from surface to volume mesh

In [1]:
import numpy
import os

import matplotlib.pyplot as plt
# import skimage
import gmsh
import sys

base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/"

folder = "VTI_shared_view/segmentations/"

mask = numpy.load(base + folder + "labels_mask.npy")

stl_file = base + folder + "mesh_repaired_surface.stl"
export_file = base + folder + "mesh_volume-form-surface"
h = 2.0

In [ ]:
def generate_tetra_mesh_from_stl(
        stl_file,
        output_file,
        element_size=2.0):

    gmsh.initialize()
    gmsh.model.add("model")

    # -------------------------
    # Import STL
    # -------------------------
    gmsh.merge(stl_file)
    

    # -------------------------
    # Create volume from surfaces
    # -------------------------
    surfaces = gmsh.model.getEntities(2)  # all 2D surfaces
    if not surfaces:
        raise RuntimeError("No surfaces found in STL.")

    surface_tags = [s[1] for s in surfaces]

    # Add surface loop
    sl = gmsh.model.geo.addSurfaceLoop(surface_tags)

    # Add volume
    gmsh.model.geo.addVolume([sl])
    gmsh.model.geo.synchronize()

    # -------------------------
    # Uniform mesh size
    # -------------------------
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", element_size)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", element_size)

    # High-quality options
    gmsh.option.setNumber("Mesh.Algorithm3D", 4)  # Delaunay
    gmsh.option.setNumber("Mesh.Optimize", 1)
    gmsh.option.setNumber("Mesh.OptimizeNetgen", 1)
    gmsh.option.setNumber("Mesh.Smoothing", 20)

    # -------------------------
    # Generate tetrahedral volume mesh
    # -------------------------
    gmsh.model.mesh.generate(3)

    # -------------------------
    # Write output 
    # -------------------------
    gmsh.write(output_file)

    gmsh.finalize()
    print("Tetra mesh written to:", output_file)


generate_tetra_mesh_from_stl(stl_file, export_file + "_1.vtk", h)


Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_repaired_surface.stl'...
Info    : 31988 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/VTI_shared_view/segmentations/mesh_repaired_surface.stl'
Info    : Meshing 1D...
Info    : Done meshing 1D (Wall 8.70787e-06s, CPU 6e-06s)
Info    : Meshing 2D...
Info    : Done meshing 2D (Wall 1.79997e-05s, CPU 1.6e-05s)
Info    : Meshing 3D...
Info    : Meshing volume 1 (Frontal)
Info    : Region 1 Face 1, 0 intersect
Info    : CalcLocalH: 15996 Points 0 Elements 31988 Surface Elements 
Info    : Check subdomain 1 / 1 
Info    : 31988 open elements 
Info    : Meshing subdomain 1 of 1 
Info    : 31988 open elements 
Info    : Use internal rules 
Info    : 31988 open elements 
Info    : Delaunay meshing 
Info    : number of points: 15996 
Info    : blockfill local h 
Info    : number of point